In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score

# Load data
employee = pd.read_csv('../data/employee_master.csv')
bench = pd.read_csv('../data/bench_allocation.csv')
finance = pd.read_csv('../data/finance_cost.csv')

# Merge into one dataset
df = employee.merge(bench, on='employee_id').merge(finance, on='employee_id')

# Encode categorical features
le_dept = LabelEncoder()
le_perf = LabelEncoder()
df['department_enc'] = le_dept.fit_transform(df['department'])
df['performance_enc'] = le_perf.fit_transform(df['performance_rating'])

# Features and target
features = ['bench_days', 'tenure_years', 'salary', 'department_enc', 'performance_enc']
X = df[features]
y = df['attrition_flag']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = LogisticRegression(random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

# Evaluate
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC Score:", round(roc_auc_score(y_test, y_prob), 3))

# Feature importance (coefficients)
importance = pd.DataFrame({'feature': features, 'coefficient': model.coef_[0]})
importance = importance.sort_values('coefficient', ascending=False)
print("\nFeature Importance:\n", importance)

              precision    recall  f1-score   support

           0       0.94      0.65      0.76       181
           1       0.15      0.58      0.23        19

    accuracy                           0.64       200
   macro avg       0.54      0.61      0.50       200
weighted avg       0.86      0.64      0.71       200

ROC-AUC Score: 0.638

Feature Importance:
            feature  coefficient
4  performance_enc     0.580289
0       bench_days     0.293380
2           salary     0.276527
1     tenure_years    -0.124453
3   department_enc    -0.186934
